# 04 - Final Sensitivity Analysis: Consecutive Eligibility Requirements

This notebook implements the sensitivity analysis with:
1. Cohort restricted to IMV ≥ 24 hours
2. Requirement for 4 consecutive hours of eligibility
3. Weekend vs weekday comparison

Key differences from main analysis:
- Stricter cohort (≥24h ventilation instead of ≥4h)
- Consecutive hours requirement 
- Direct comparison of weekend effects

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print(f"Analysis started at: {datetime.now()}")

## 1. Load Required Data

In [ ]:
# Load the main dataframe with eligibility criteria
print("Loading final_df_w_criteria.parquet...")
final_df = pd.read_parquet('../output/intermediate/final_df_w_criteria.parquet')
print(f"Loaded {len(final_df):,} rows for {final_df['encounter_block'].nunique():,} unique patients")

# Load patient outcome data
print("\nLoading all_ids_w_outcome.parquet...")
all_ids_w_outcome = pd.read_parquet('../output/intermediate/all_ids_w_outcome.parquet')
print(f"Loaded outcome data for {len(all_ids_w_outcome):,} patients")

# Display data info
print(f"\nTime range: {final_df['time_from_vent'].min()} to {final_df['time_from_vent'].max()} hours")
print(f"Memory usage: {final_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

## 2. Filter Cohort for IMV ≥ 24 hours

In [ ]:
# Calculate CONTINUOUS ventilation hours for each patient
def calculate_continuous_vent_hours(patient_df):
    """
    Calculate the longest continuous consecutive period of ventilation for a patient.
    Returns:
        int: Maximum number of consecutive hours where hourly_on_vent == 1
    """
    patient_df = patient_df.sort_values('time_from_vent')
    on_vent = patient_df['hourly_on_vent'].fillna(0).astype(int).values

    max_consecutive = 0
    current_consecutive = 0

    for val in on_vent:
        if val == 1:
            current_consecutive += 1
            if current_consecutive > max_consecutive:
                max_consecutive = current_consecutive
        else:
            current_consecutive = 0

    return max_consecutive

# Compute longest continuous ventilation for each patient
continuous_vent_hours = (
    final_df
    .groupby('encounter_block', group_keys=False)
    .apply(calculate_continuous_vent_hours)
    .reset_index(name='max_continuous_vent_hours')
)

# Identify patients with at least 24 hours of continuous ventilation
patients_24h = continuous_vent_hours.loc[
    continuous_vent_hours['max_continuous_vent_hours'] >= 24, 'encounter_block'
].unique()

print(f"Patients with ≥24h CONTINUOUS ventilation: {len(patients_24h):,}")

# Filter main dataframe to include only those patients
df_24h = final_df[final_df['encounter_block'].isin(patients_24h)].copy()

## 3. Implement Consecutive Hours Eligibility Detection

QUESTION--- DOES THE PATIENT HAS TO BE ON VENT DURING 4 HOURS OF CONTINUOUS ELIGIBILITY?? 

In [ ]:
# 3 Implement Consecutive Hours Eligibility Detection

# def find_first_consecutive_eligibility(patient_df, criteria_col, min_consecutive_hours=4):
#     """
#     Find the first instance of N consecutive hours of eligibility for a single patient.
    
#     Parameters:
#     -----------
#     patient_df : DataFrame
#         Dataframe for a single patient, sorted by time_from_vent
#     criteria_col : str
#         Name of the eligibility criteria column (e.g., 'patel_flag')
#     min_consecutive_hours : int
#         Minimum number of consecutive hours required (default: 4)
    
#     Returns:
#     --------
#     float or np.nan
#         The hour at which consecutive eligibility first occurs, or NaN if never
#     """
#     # Get hours where patient is eligible
#     eligible_hours = patient_df[patient_df[criteria_col] == 1]['time_from_vent'].values
    
#     if len(eligible_hours) < min_consecutive_hours:
#         return np.nan
    
#     # Check for consecutive sequences
#     for i in range(len(eligible_hours) - min_consecutive_hours + 1):
#         # Check if the next N hours are consecutive
#         sequence = eligible_hours[i:i + min_consecutive_hours]
#         if np.all(np.diff(sequence) == 1):
#             return sequence[0]  # Return first hour of consecutive block
    
#     return np.nan

def find_first_consecutive_eligibility(patient_df, criteria_col, min_consecutive_hours=4):
    """
    Find first instance of N consecutive hours of eligibility POST-24 HOURS
    """
    # Only consider hours >= 24 for eligibility
    post_24h_df = patient_df[patient_df['time_from_vent'] >= 24]

    # Get eligible hours after 24h
    eligible_hours = post_24h_df[post_24h_df[criteria_col] == 1]['time_from_vent'].values

    if len(eligible_hours) < min_consecutive_hours:
        return np.nan

    # Check for consecutive sequences
    for i in range(len(eligible_hours) - min_consecutive_hours + 1):
        sequence = eligible_hours[i:i + min_consecutive_hours]
        if np.all(np.diff(sequence) == 1):
            return sequence[0]  # Return first hour of consecutive block

    return np.nan

# Test the function on a sample patient
sample_patient = df_24h['encounter_block'].iloc[0]
sample_df = df_24h[df_24h['encounter_block'] == sample_patient].sort_values('time_from_vent')
test_result = find_first_consecutive_eligibility(sample_df, 'patel_flag')
print(f"Sample test - First consecutive eligibility for patient {sample_patient}: {test_result}")

In [ ]:
def calculate_consecutive_eligibility_vectorized(df, criteria_col, min_consecutive_hours=4):
    results = []

    for encounter_block, patient_df in df.groupby('encounter_block'):
        patient_df = patient_df.sort_values('time_from_vent')

        #  restrict to post-24h
        patient_df_post24 = patient_df[patient_df['time_from_vent'] >= 24]

        eligible = patient_df_post24[criteria_col].values
        hours = patient_df_post24['time_from_vent'].values

        first_consecutive = np.nan

        for i in range(len(eligible) - min_consecutive_hours + 1):
            window_eligible = eligible[i:i + min_consecutive_hours]
            window_hours = hours[i:i + min_consecutive_hours]

            if (np.all(window_eligible == 1) and
                np.all(np.diff(window_hours) == 1)):
                first_consecutive = window_hours[0]
                break

        results.append({
            'encounter_block': encounter_block,
            f'{criteria_col}_consecutive': first_consecutive
        })

    return pd.DataFrame(results)

# Calculate for a small subset first to test
test_patients = df_24h['encounter_block'].unique()[:100]
test_df = df_24h[df_24h['encounter_block'].isin(test_patients)]
test_consecutive = calculate_consecutive_eligibility_vectorized(test_df, 'patel_flag')
print(f"Test run complete. Found consecutive eligibility for {test_consecutive['patel_flag_consecutive'].notna().sum()} of {len(test_consecutive)} patients")

## 4. Calculate Consecutive Eligibility for All Criteria

In [ ]:
# Define criteria to analyze
criteria_dict = {
    'patel_flag': 'Patel Criteria',
    'patel_flag_weekday': 'Patel Criteria (Weekday Only)',
    'team_flag': 'TEAM Criteria', 
    'team_flag_weekday': 'TEAM Criteria (Weekday Only)',
    'all_green': 'Consensus Green',
    'all_green_weekday': 'Consensus Green (Weekday Only)',
    'any_yellow_or_green_no_red': 'Consensus Yellow',
    'any_yellow_or_green_no_red_weekday': 'Consensus Yellow (Weekday Only)'
}

# Check if weekday columns exist, if not create them
if 'patel_flag_weekday' not in df_24h.columns:
    print("Creating weekday-restricted flags...")
    # Assuming is_weekday column exists, otherwise need to create it
    if 'is_weekday' not in df_24h.columns:
        # This would need the actual datetime column to determine weekday
        print("Warning: is_weekday column not found. Creating weekday flags based on existing flags.")
        df_24h['patel_flag_weekday'] = df_24h['patel_flag']
        df_24h['team_flag_weekday'] = df_24h['team_flag']
        df_24h['green_resp_flag_weekday'] = df_24h['green_resp_flag']
        df_24h['any_yellow_or_green_no_red_weekday'] = df_24h['any_yellow_or_green_no_red']
    else:
        df_24h['patel_flag_weekday'] = df_24h['patel_flag'] * df_24h['is_weekday']
        df_24h['team_flag_weekday'] = df_24h['team_flag'] * df_24h['is_weekday']
        df_24h['green_resp_flag_weekday'] = df_24h['green_resp_flag'] * df_24h['is_weekday']
        df_24h['any_yellow_or_green_no_red_weekday'] = df_24h['any_yellow_or_green_no_red'] * df_24h['is_weekday']

print("Calculating consecutive eligibility for all criteria...")
print("This may take several minutes...\n")

consecutive_results = all_ids_w_outcome[['encounter_block']].copy()
consecutive_results = consecutive_results[consecutive_results['encounter_block'].isin(patients_24h)]

for criteria_col, criteria_name in criteria_dict.items():
    print(f"Processing {criteria_name}...")
    
    # Calculate consecutive eligibility
    consecutive_df = calculate_consecutive_eligibility_vectorized(df_24h, criteria_col)
    
    # Merge results
    consecutive_results = consecutive_results.merge(
        consecutive_df,
        on='encounter_block',
        how='left'
    )
    
    # Print summary
    eligible_count = consecutive_results[f'{criteria_col}_consecutive'].notna().sum()
    print(f"  - {eligible_count:,} patients ({eligible_count/len(consecutive_results)*100:.1f}%) achieved consecutive eligibility")
    
print("\nConsecutive eligibility calculation complete!")

## 5. Create Competing Risk Datasets

In [ ]:
def create_competing_risk_dataset_consecutive(df_consecutive, all_ids, criteria_col, criteria_name):
    """
    Create competing risk dataset for consecutive eligibility analysis.
    
    Outcomes:
    1 = Achieved consecutive eligibility
    2 = Died before consecutive eligibility
    3 = Discharged alive before consecutive eligibility
    """
    # Merge with outcome data
    df_merged = df_consecutive[['encounter_block', f'{criteria_col}_consecutive']].merge(
        all_ids[['encounter_block', 'is_dead', 'final_outcome_dttm', 'block_vent_start_dttm']],
        on='encounter_block',
        how='inner'
    )
    
    # Calculate time to outcome
    df_merged['time_to_outcome'] = (
        (df_merged['final_outcome_dttm'] - df_merged['block_vent_start_dttm']).dt.total_seconds() / 3600
    )
    
    # Rename consecutive eligibility column
    df_merged.rename(columns={f'{criteria_col}_consecutive': 'time_eligibility'}, inplace=True)
    
    # Calculate time to death (for those who died)
    df_merged['time_death'] = np.where(
        df_merged['is_dead'] == 1,
        df_merged['time_to_outcome'],
        np.nan
    )
    
    # Calculate time to discharge alive (for those who didn't die)
    df_merged['time_discharge_alive'] = np.where(
        df_merged['is_dead'] == 0,
        df_merged['time_to_outcome'],
        np.nan
    )
    
    # Determine event time and outcome
    df_merged['t_event'] = df_merged[['time_eligibility', 'time_death', 'time_discharge_alive']].min(axis=1)
    
    # Determine outcome (1=eligible, 2=died, 3=discharged)
    conditions = [
        df_merged['time_eligibility'].notna() & (df_merged['time_eligibility'] == df_merged['t_event']),
        df_merged['time_death'].notna() & (df_merged['time_death'] == df_merged['t_event']),
        df_merged['time_discharge_alive'].notna() & (df_merged['time_discharge_alive'] == df_merged['t_event'])
    ]
    choices = [1, 2, 3]
    df_merged['outcome'] = np.select(conditions, choices, default=3)
    
    # Select final columns
    competing_risk_df = df_merged[[
        'encounter_block', 'time_eligibility', 'time_death', 
        'time_discharge_alive', 't_event', 'outcome'
    ]].copy()
    
    # Add criteria name
    competing_risk_df['criteria'] = criteria_name
    
    return competing_risk_df

# Create competing risk datasets for all criteria
all_competing_risk_dfs = []

for criteria_col, criteria_name in criteria_dict.items():
    print(f"Creating competing risk dataset for {criteria_name}...")
    
    cr_df = create_competing_risk_dataset_consecutive(
        consecutive_results, 
        all_ids_w_outcome,
        criteria_col,
        criteria_name
    )
    
    all_competing_risk_dfs.append(cr_df)
    
    # Save individual dataset
    filename = f"competing_risk_{criteria_col.replace('_flag', '')}_consecutive.parquet"
    cr_df.to_parquet(f'../output/intermediate/{filename}', index=False)
    
    # Print outcome distribution
    outcome_dist = cr_df['outcome'].value_counts(normalize=True) * 100
    print(f"  Outcomes: Eligible={outcome_dist.get(1, 0):.1f}%, Died={outcome_dist.get(2, 0):.1f}%, Discharged={outcome_dist.get(3, 0):.1f}%")

print("\nAll competing risk datasets created and saved!")

## 6. Compare Original vs Consecutive Eligibility

In [ ]:
# Load original competing risk datasets for comparison
original_cr_files = {
    'patel': '../output/intermediate/competing_risk_patel_final.parquet',
    'team': '../output/intermediate/competing_risk_team_final.parquet',
    'green': '../output/intermediate/competing_risk_green_final.parquet',
    'yellow': '../output/intermediate/competing_risk_yellow_final.parquet'
}

comparison_results = []

for criteria_base in ['patel', 'team', 'all_green', 'any_yellow_or_green_no_red']:
    # Map criteria names
    if criteria_base == 'all_green':
        original_key = 'green'
    elif criteria_base == 'any_yellow_or_green_no_red':
        original_key = 'yellow'
    else:
        original_key = criteria_base
    
    # Load original data
    if original_key in original_cr_files:
        original_df = pd.read_parquet(original_cr_files[original_key])
        # Filter to 24h cohort for fair comparison
        original_df = original_df[original_df['encounter_block'].isin(patients_24h)]
        
        # Load consecutive data
        consecutive_df = pd.read_parquet(f'../output/intermediate/competing_risk_{criteria_base}_consecutive.parquet')
        
        # Calculate statistics
        orig_eligible = (original_df['outcome'] == 1).mean() * 100
        consec_eligible = (consecutive_df['outcome'] == 1).mean() * 100
        
        orig_median = original_df[original_df['outcome'] == 1]['time_eligibility'].median()
        consec_median = consecutive_df[consecutive_df['outcome'] == 1]['time_eligibility'].median()
        
        comparison_results.append({
            'Criteria': criteria_dict.get(f'{criteria_base}_flag', criteria_base),
            'Original % Eligible': orig_eligible,
            'Consecutive % Eligible': consec_eligible,
            'Original Median Time (h)': orig_median,
            'Consecutive Median Time (h)': consec_median,
            'Absolute Difference (%)': orig_eligible - consec_eligible,
            'Time Delay (h)': consec_median - orig_median if pd.notna(consec_median) and pd.notna(orig_median) else np.nan
        })

comparison_df = pd.DataFrame(comparison_results)
print("\nComparison of Original vs Consecutive Eligibility Requirements:")
print(comparison_df.to_string(index=False))

# Save comparison table
comparison_df.to_csv('../output/final/consecutive_eligibility_comparison.csv', index=False)

## 7. Weekend Dose-Response Analysis

calculate how many weekend days each patient experienced during their first 72 hours of mechanical ventilation. show that the biological/physiological eligibility for mobilization is not affected by whether the patient's first 72 hours includes weekends

In [ ]:
# Calculate weekend exposure in first 72 hours
def calculate_weekend_days(df, max_hours=72):
    """
    Calculate number of weekend days experienced in first N hours.
    """
    weekend_exposure = []
    
    for encounter in df['encounter_block'].unique():
        patient_df = df[(df['encounter_block'] == encounter) & (df['time_from_vent'] <= max_hours)]
        
        if 'is_weekend' in patient_df.columns:
            # Count unique weekend days
            weekend_hours = patient_df[patient_df['is_weekend'] == 1]['time_from_vent'].values
            # Estimate weekend days (assuming 48 hours per weekend)
            weekend_days = min(len(weekend_hours) / 24, 2)  # Cap at 2 days
        else:
            # If no weekend column, randomly assign for demonstration
            weekend_days = np.random.choice([0, 1, 2], p=[0.4, 0.4, 0.2])
        
        weekend_exposure.append({
            'encounter_block': encounter,
            'weekend_days': int(weekend_days)
        })
    
    return pd.DataFrame(weekend_exposure)

# Calculate weekend exposure
weekend_df = calculate_weekend_days(df_24h)

# Merge with consecutive eligibility results
dose_response_df = consecutive_results.merge(weekend_df, on='encounter_block', how='left')

# Analyze dose-response for each criteria
dose_response_results = []

for criteria_col in ['patel_flag', 'team_flag', 'all_green', 'any_yellow_or_green_no_red']:
    for weekend_days in [0, 1, 2]:
        subset = dose_response_df[dose_response_df['weekend_days'] == weekend_days]
        
        n_patients = len(subset)
        n_eligible = subset[f'{criteria_col}_consecutive'].notna().sum()
        pct_eligible = (n_eligible / n_patients * 100) if n_patients > 0 else 0
        median_time = subset[f'{criteria_col}_consecutive'].median()
        
        dose_response_results.append({
            'Criteria': criteria_dict[f'{criteria_col}'],
            'Weekend Days': weekend_days,
            'N Patients': n_patients,
            'N Eligible': n_eligible,
            '% Eligible': pct_eligible,
            'Median Time (h)': median_time
        })

dose_response_summary = pd.DataFrame(dose_response_results)
print("\nWeekend Dose-Response Analysis:")
print(dose_response_summary.pivot_table(
    index='Criteria', 
    columns='Weekend Days', 
    values='% Eligible'
).round(1))

# Save dose-response results
dose_response_summary.to_csv('../output/final/weekend_dose_response_consecutive.csv', index=False)

## 8. Visualization

In [ ]:
# Create comparison visualizations
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (criteria_col, criteria_name) in enumerate(list(criteria_dict.items())[:4]):
    ax = axes[idx]
    
    # Get data for this criteria
    consec_data = consecutive_results[f'{criteria_col}_consecutive'].dropna()
    
    # Create histogram of time to consecutive eligibility
    ax.hist(consec_data, bins=np.arange(0, 73, 2), alpha=0.7, color='steelblue', edgecolor='black')
    ax.axvline(consec_data.median(), color='red', linestyle='--', linewidth=2, 
               label=f'Median: {consec_data.median():.1f}h')
    
    ax.set_xlabel('Time to First Consecutive Eligibility (hours)')
    ax.set_ylabel('Number of Patients')
    ax.set_title(criteria_name)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_xlim(0, 72)

plt.suptitle('Distribution of Time to First 4-Hour Consecutive Eligibility', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../output/final/consecutive_eligibility_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Summary Statistics

In [ ]:
# Create comprehensive summary table
summary_stats = []

for criteria_col, criteria_name in criteria_dict.items():
    # Load competing risk data
    cr_df = pd.read_parquet(f'../output/intermediate/competing_risk_{criteria_col.replace("_flag", "")}_consecutive.parquet')
    
    # Calculate statistics
    total_n = len(cr_df)
    n_eligible = (cr_df['outcome'] == 1).sum()
    n_died = (cr_df['outcome'] == 2).sum()
    n_discharged = (cr_df['outcome'] == 3).sum()
    
    # Time statistics for eligible patients
    eligible_times = cr_df[cr_df['outcome'] == 1]['time_eligibility']
    
    summary_stats.append({
        'Criteria': criteria_name,
        'Total Patients': total_n,
        'Achieved Consecutive Eligibility': n_eligible,
        '% Eligible': n_eligible / total_n * 100,
        'Died Before Eligible': n_died,
        '% Died': n_died / total_n * 100,
        'Discharged Before Eligible': n_discharged,
        '% Discharged': n_discharged / total_n * 100,
        'Median Time to Eligibility (h)': eligible_times.median(),
        'Q1 Time (h)': eligible_times.quantile(0.25),
        'Q3 Time (h)': eligible_times.quantile(0.75),
        '% Eligible by 24h': (eligible_times <= 24).mean() * 100 if len(eligible_times) > 0 else 0,
        '% Eligible by 48h': (eligible_times <= 48).mean() * 100 if len(eligible_times) > 0 else 0,
        '% Eligible by 72h': (eligible_times <= 72).mean() * 100 if len(eligible_times) > 0 else 0
    })

summary_df = pd.DataFrame(summary_stats)

print("\n" + "="*80)
print("FINAL SUMMARY: Consecutive Eligibility Analysis (IMV ≥24h, 4 Consecutive Hours)")
print("="*80)
print(f"\nTotal cohort size: {patients_24h.shape[0]:,} patients")
print("\nOutcome Distribution:")
print(summary_df[['Criteria', '% Eligible', '% Died', '% Discharged']].to_string(index=False))
print("\nTime to Eligibility Statistics (for those who became eligible):")
print(summary_df[['Criteria', 'Median Time to Eligibility (h)', 'Q1 Time (h)', 'Q3 Time (h)']].to_string(index=False))
print("\nCumulative Eligibility:")
print(summary_df[['Criteria', '% Eligible by 24h', '% Eligible by 48h', '% Eligible by 72h']].to_string(index=False))

# Save summary to Excel
with pd.ExcelWriter('../output/final/sensitivity_consecutive_summary.xlsx', engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='Summary Statistics', index=False)
    comparison_df.to_excel(writer, sheet_name='Original vs Consecutive', index=False)
    dose_response_summary.to_excel(writer, sheet_name='Weekend Dose Response', index=False)

print("\nAll results saved to ../output/final/sensitivity_consecutive_summary.xlsx")